# Trabalho 1, Tema 1: Ciclone → maré de tempestade e ondas

**ACA0430 — Meteorologia Sinótica e Aplicações à Oceanografia (IAG/USP), 2º semestre de 2026**

**Objetivo.** Montar as figuras de base de um estudo de caso em que um **ciclone extratropical** no Atlântico Sul gera **maré meteorológica** (sobre-elevação do nível do mar) e um **campo de ondas** intenso na costa S/SE do Brasil. O exemplo padrão é a **ressaca de Santos de 21/08/2016**.

**Ligação com as aulas.** O vento sobre o oceano transfere momento para a camada de Ekman. No **Hemisfério Sul o transporte de Ekman é para a esquerda do vento**. Na costa SE do Brasil, que corre de SW para NE com o oceano a SE:

- **vento de SW**, paralelo à costa, gera transporte de Ekman **para a costa**: a água se acumula e o nível sobe (**maré meteorológica positiva**, ressaca);
- **vento de NE** gera transporte **para fora da costa**: o nível desce (**maré meteorológica negativa**).

Ao mesmo tempo, o vento forte e persistente sobre uma pista longa gera **ondas** altas que chegam à costa. Campos et al. (2010) caracterizam os eventos extremos de nível do mar em Santos.

**Este notebook é um ponto de partida**, não uma receita fechada: você pode (e deve) mudar o período, a região e acrescentar outros dados.

**Dados:** ERA5 (atmosfera, horário, 0,25°), GLORYS12 (oceano, diário, 1/12°) e WAVERYS (ondas, 3-horário, 0,2°), todos lidos de armazenamentos **Zarr na nuvem** (ARCO), sem baixar arquivos inteiros.

## 1. Instalação dos pacotes

O Colab já traz `numpy`, `pandas`, `xarray` e `matplotlib`. Os pacotes abaixo **não vêm pré-instalados**:

- `copernicusmarine`: *toolbox* oficial do Copernicus Marine (GLORYS12 e WAVERYS);
- `gcsfs` e `zarr`: leitura do ERA5 direto do Google Cloud Storage, em formato Zarr;
- `cartopy`: mapas com linha de costa e projeções;
- `cmocean`: paletas de cores pensadas para oceanografia.

A instalação leva cerca de 1 minuto. Em geral **não é preciso reiniciar** a sessão. Se aparecer erro de importação na célula seguinte, use *Ambiente de execução → Reiniciar sessão* e rode o notebook de novo a partir daqui.

Fora do Colab (no seu computador), a instalação automática é pulada: instale os pacotes no seu ambiente antes.

In [ ]:
import sys

# Detecta se estamos no Google Colab
EM_COLAB = "google.colab" in sys.modules

# Pacotes que não vêm pré-instalados no Colab
if EM_COLAB:
    !pip install -q copernicusmarine gcsfs zarr cartopy cmocean

## 2. Importações

Importamos tudo o que vamos usar e imprimimos as versões dos pacotes principais. **Se algo der errado, copie esta saída ao pedir ajuda**: muitos problemas vêm de diferenças de versão.

In [ ]:
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import zarr
import gcsfs
import copernicusmarine

warnings.filterwarnings("ignore", category=FutureWarning)

for nome, modulo in [("xarray", xr), ("zarr", zarr), ("gcsfs", gcsfs),
                     ("copernicusmarine", copernicusmarine), ("cartopy", cartopy)]:
    print(f"{nome:18s} {modulo.__version__}")

## 3. Credenciais do Copernicus Marine (CMEMS)

O **ERA5** que usamos aqui é **público**: não precisa de conta. Já o **CMEMS** (GLORYS12 e WAVERYS) exige uma **conta gratuita**.

**Nunca escreva sua senha no notebook.** Se você compartilhar o notebook, a senha vai junto.

### Passo a passo (uma vez só)

1. **Crie a conta** em https://data.marine.copernicus.eu/register. Preencha nome, e-mail, usuário e senha e confirme pelo link que chega no e-mail. Guarde o **nome de usuário**, que não é o e-mail.
2. **Crie os segredos no Colab:**
   - na barra lateral esquerda do Colab, clique no ícone de **chave** ("Secrets" / "Segredos");
   - clique em **"+ Adicionar novo segredo"**;
   - em *Nome*, escreva `COPERNICUSMARINE_SERVICE_USERNAME` e, em *Valor*, seu usuário do Copernicus Marine;
   - adicione um segundo segredo, `COPERNICUSMARINE_SERVICE_PASSWORD`, com a sua senha;
   - ative a chave **"Acesso ao notebook"** nos dois segredos.
3. Rode a célula abaixo. Se o Colab perguntar se o notebook pode acessar os segredos, clique em **"Conceder acesso"**.

Se você não criar os segredos, a célula pede usuário e senha numa caixa de texto (`getpass`), que não fica gravada no notebook.

Os segredos ficam na sua conta Google e servem para todos os seus notebooks do Colab.

In [ ]:
usuario = os.environ.get("COPERNICUSMARINE_SERVICE_USERNAME")
senha = os.environ.get("COPERNICUSMARINE_SERVICE_PASSWORD")

# 1) Segredos do Colab (ícone de chave na barra lateral)
if EM_COLAB and not usuario:
    from google.colab import userdata
    try:
        usuario = userdata.get("COPERNICUSMARINE_SERVICE_USERNAME")
        senha = userdata.get("COPERNICUSMARINE_SERVICE_PASSWORD")
    except Exception:  # segredo não criado ou acesso não concedido
        usuario, senha = None, None

if not usuario and copernicusmarine.login(check_credentials_valid=True):
    # Fora do Colab: login já salvo neste computador (comando `copernicusmarine login`)
    print("Usando o login do Copernicus Marine já salvo neste computador.")
else:
    # 2) Alternativa: perguntar usuário e senha (a senha não aparece na tela)
    if not usuario:
        from getpass import getpass
        usuario = input("Usuário do Copernicus Marine: ")
        senha = getpass("Senha do Copernicus Marine: ")

    # O toolbox também lê estas variáveis de ambiente
    os.environ["COPERNICUSMARINE_SERVICE_USERNAME"] = usuario
    os.environ["COPERNICUSMARINE_SERVICE_PASSWORD"] = senha

    # 3) Login (uma vez por sessão)
    ok = copernicusmarine.login(username=usuario, password=senha, force_overwrite=True)
    print("Login OK" if ok else "Falha no login: confira usuário e senha")

del senha  # não deixa a senha numa variável visível

## 4. Parâmetros do estudo de caso

**Esta é a única célula que você precisa alterar** para estudar outro evento. Todo o resto do notebook usa estes valores.

- `REGIAO`: limites do domínio em graus (longitude negativa a oeste). O padrão (44°S–20°S, 65°W–35°W) inclui o sul da América do Sul, para mostrar a **gênese do ciclone**.
- `INICIO`, `FIM`: janela do evento (UTC), usada nos mapas.
- `PASSO_MAPAS_H`: intervalo entre mapas, em horas. Use múltiplos de 3, porque o WAVERYS é 3-horário.
- `MAPAS_INICIO`, `MAPAS_FIM`: subconjunto da janela que vai para os mapas. Por padrão, a janela inteira.
- `SERIE_INICIO`, `SERIE_FIM`: período da série temporal no ponto. Pode ser **mais longo** que o evento, para mostrar o antes e o depois.
- `PONTO`: local da série temporal (ao largo de Santos).
- `ANGULO_COSTA_GRAUS`: orientação da costa no ponto (ver a convenção abaixo).

**Convenção do ângulo da costa.** É o azimute (graus a partir do norte, sentido horário) de uma direção **ao longo da costa escolhida de forma que o oceano fique à DIREITA** de quem caminha nela. Em Santos, a costa corre aproximadamente para ENE com o oceano a SSE: cerca de 55°. A componente do vento **paralela à costa** é positiva quando o vento sopra **nesse sentido** (vento de SW, em Santos). No Hemisfério Sul, essa componente positiva produz transporte de Ekman para a costa, ou seja, sobre-elevação.

```
          N
          ↑        ↗ costa (55°)
          |      /
   terra  |    /     oceano à direita
          |  /
          |/__________→ L
```

In [ ]:
# >>> ALUNO: região do domínio (graus; oeste e sul negativos)
REGIAO = dict(lon_o=-65.0, lon_l=-35.0, lat_s=-44.0, lat_n=-20.0)

# >>> ALUNO: janela do evento (UTC)
INICIO = "2016-08-15T00:00"
FIM = "2016-08-23T21:00"

# >>> ALUNO: mapas a cada PASSO_MAPAS_H horas, entre MAPAS_INICIO e MAPAS_FIM
PASSO_MAPAS_H = 6
MAPAS_INICIO = INICIO
MAPAS_FIM = FIM

# >>> ALUNO: período da série temporal no ponto (pode ser mais longo que o evento)
SERIE_INICIO = "2016-08-14T00:00"
SERIE_FIM = "2016-08-25T21:00"

# >>> ALUNO: ponto da série temporal e orientação da costa nesse ponto
PONTO = dict(lon=-46.30, lat=-24.30, nome="Largo de Santos")
ANGULO_COSTA_GRAUS = 55.0

# >>> ALUNO (opcional): período de referência para a anomalia do nível do mar (GLORYS).
# None = usar a média do próprio período do evento (INICIO a FIM).
ZOS_REF_INICIO = "2016-01-01"   # ex.: None, "2016-07-01"
ZOS_REF_FIM = "2016-12-31"      # ex.: None, "2016-09-30"

# Instantes dos mapas
DATAS_MAPAS = pd.date_range(MAPAS_INICIO, MAPAS_FIM, freq=f"{PASSO_MAPAS_H}h")
print(f"{len(DATAS_MAPAS)} mapas, de {DATAS_MAPAS[0]} a {DATAS_MAPAS[-1]}")

### Pastas de saída

No Colab, os recortes de dados (NetCDF) vão para `/content/dados/` e as figuras para `/content/figuras/`. **Atenção: a pasta `/content` é apagada quando a sessão termina.** No fim do notebook há uma célula opcional para copiar tudo para o seu Google Drive.

In [ ]:
PASTA_BASE = Path("/content") if EM_COLAB else Path.cwd() / "saida"
PASTA_DADOS = PASTA_BASE / "dados"
PASTA_FIGURAS = PASTA_BASE / "figuras"
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)
print("Dados:  ", PASTA_DADOS)
print("Figuras:", PASTA_FIGURAS)

## 5. Funções auxiliares

Poucas funções, usadas várias vezes ao longo do notebook:

- `lon_para_360` / `lon_para_180`: o ERA5 usa longitude de **0 a 360°** (o Brasil fica entre \~305° e \~330°). Recortamos com a longitude convertida e só voltamos para −180..180 **depois** do recorte, na hora de plotar;
- `checar_tamanho`: mostra quantos MB o recorte vai ocupar **antes** de carregar;
- `mapa_base`: costa, fronteiras, grade e (opcionalmente) continente em cinza;
- `passo_vetores`: reduz o número de setas para o mapa não virar um borrão;
- `ponto_oceano_mais_proximo`: perto da costa, a grade dos modelos oceânicos tem pontos de terra (NaN). Esta função escolhe o ponto **de oceano** mais próximo;
- `rodape` e `salvar_figura`: fonte dos dados e exportação em PNG.

In [ ]:
def lon_para_360(lon):
    """Converte longitude de -180..180 para 0..360 (convenção do ERA5)."""
    return lon % 360


def lon_para_180(ds, nome="longitude"):
    """Converte a coordenada de longitude de 0..360 para -180..180 e reordena (para plotar)."""
    ds = ds.assign_coords({nome: ((ds[nome] + 180) % 360) - 180})
    return ds.sortby(nome)


def checar_tamanho(ds, nome, limite_mb=1500):
    """Imprime o tamanho estimado do recorte ANTES de carregar na memória."""
    mb = ds.nbytes / 1e6
    print(f"{nome}: {mb:.1f} MB")
    if mb > limite_mb:
        print(f"  ATENÇÃO: mais de {limite_mb} MB! Reduza o período ou a área antes de carregar.")
    return mb


def mapa_base(ax, regiao, terra=True):
    """Prepara um eixo cartopy: extensão, costa, grade com rótulos.

    terra=True pinta os continentes de cinza POR CIMA dos campos (para campos oceânicos).
    """
    ax.set_extent([regiao["lon_o"], regiao["lon_l"], regiao["lat_s"], regiao["lat_n"]],
                  crs=ccrs.PlateCarree())
    if terra:
        ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=3)
    ax.coastlines("50m", linewidth=0.6, zorder=4)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3, edgecolor="0.4", zorder=4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.5", linestyle="--", zorder=5)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}
    return ax


def passo_vetores(n_pontos, n_setas=20):
    """Quantos pontos pular para desenhar ~n_setas vetores por eixo."""
    return max(1, n_pontos // n_setas)


def barra_cores(fig, mapeavel, rotulo):
    """Ajusta o espaço entre os painéis e põe UMA barra de cores horizontal embaixo da figura.

    Devolve a altura (fração da figura) da barra, útil para posicionar a seta de referência.
    """
    h = fig.get_figheight()
    y_barra = 0.8 / h
    fig.subplots_adjust(left=0.04, right=0.98, top=1 - 0.4 / h, bottom=y_barra + 1.0 / h,
                        wspace=0.18, hspace=0.3)
    cax = fig.add_axes([0.25, y_barra, 0.5, 0.15 / h])
    fig.colorbar(mapeavel, cax=cax, orientation="horizontal", label=rotulo)
    return y_barra


def rodape(fig, texto):
    """Escreve a fonte dos dados no rodapé da figura."""
    fig.text(0.01, 0.005, texto, fontsize=7, color="0.3", ha="left", va="bottom")


def ponto_oceano_mais_proximo(campo, lon, lat, nome_lon="longitude", nome_lat="latitude"):
    """Ponto de grade mais próximo de (lon, lat) que NÃO é terra (valor não-NaN).

    `campo` é um DataArray 2D (lat, lon) em que a terra está como NaN.
    Devolve (lon, lat) do ponto escolhido.
    """
    coslat = np.cos(np.deg2rad(lat))
    dist2 = ((campo[nome_lon] - lon) * coslat) ** 2 + (campo[nome_lat] - lat) ** 2
    dist2 = dist2.where(campo.notnull())
    idx = dist2.argmin(dim=[nome_lat, nome_lon])
    lon_ok = float(campo[nome_lon][idx[nome_lon]])
    lat_ok = float(campo[nome_lat][idx[nome_lat]])
    return lon_ok, lat_ok


def salvar_figura(fig, nome):
    """Salva a figura em PNG (200 dpi) na pasta de figuras."""
    caminho = PASTA_FIGURAS / nome
    fig.savefig(caminho, dpi=200, bbox_inches="tight")
    print("Figura salva:", caminho)

## 6. ERA5: abertura do armazenamento ARCO (Zarr na nuvem)

O **ARCO-ERA5** é uma cópia do ERA5 no Google Cloud em formato **Zarr**, *Analysis-Ready, Cloud-Optimized* (Abernathey et al., 2021). Em vez de baixar arquivos inteiros, abrimos o conjunto completo (1940 até hoje, horário, 0,25°) de forma **preguiçosa**: só os metadados são lidos agora. Os números só são transferidos quando chamamos `.load()`.

O armazenamento é dividido em pedaços (*chunks*) de **um instante × globo inteiro**, ótimo para **mapas**. Por isso:

- abrimos o armazenamento **uma única vez** (leva alguns segundos);
- recortamos **no espaço e no tempo antes** de carregar;
- nunca fazemos um laço hora a hora reabrindo o armazenamento.

O ERA5 "final" tem cerca de 3 meses de atraso. Os meses mais recentes são o **ERA5T** (preliminar), que pode ser corrigido depois.

In [ ]:
URL_ERA5 = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

era5 = xr.open_zarr(URL_ERA5, chunks=None, storage_options=dict(token="anon"))

print("ERA5 final:          ", era5.attrs["valid_time_start"], "a", era5.attrs["valid_time_stop"])
print("ERA5T (preliminar) até", era5.attrs["valid_time_stop_era5t"])
print("Número de variáveis: ", len(era5.data_vars))

### Nomes das variáveis

As variáveis do ARCO-ERA5 têm **nomes longos** (`mean_sea_level_pressure`, `10m_u_component_of_wind`...). Os nomes curtos do ECMWF (`msl`, `u10`) só aparecem nos atributos. A célula abaixo lista as variáveis com seus nomes curtos e unidades. Use-a para procurar outras variáveis que queira acrescentar.

In [ ]:
# >>> ALUNO: troque as palavras-chave para procurar outras variáveis
PALAVRAS = ["sea_level", "10m_", "sea_surface", "heat_flux", "radiation", "evaporation",
            "precipitation", "stress", "geopotential", "land_sea"]

for v in sorted(era5.data_vars):
    if any(p in v for p in PALAVRAS):
        atr = era5[v].attrs
        print(f"{v:58s} {atr.get('short_name', '?'):10s} [{atr.get('units', '?')}]")

### Recorte do ERA5

Variáveis usadas neste tema: **pressão ao nível do mar** (`mean_sea_level_pressure`, em Pa), **componentes u e v do vento a 10 m** (m s⁻¹) e a **máscara terra-mar** (`land_sea_mask`), usada para escolher pontos de oceano.

Cuidados no recorte:

1. **Latitude decrescente** (90 → −90): o `slice` vai do norte para o sul.
2. **Longitude 0–360**: convertemos os limites com `lon_para_360` (região que cruza o meridiano de Greenwich exigiria dois recortes, o que não é o caso do Brasil).
3. Recortamos **um único período**, que cobre tanto os mapas quanto a série temporal, para ler o armazenamento uma vez só.

In [ ]:
VARS_ERA5 = {
    "mean_sea_level_pressure": "msl",
    "10m_u_component_of_wind": "u10",
    "10m_v_component_of_wind": "v10",
    "land_sea_mask": "lsm",
}

# Período que cobre os mapas e a série temporal
e5_inicio = min(pd.Timestamp(INICIO), pd.Timestamp(SERIE_INICIO))
e5_fim = max(pd.Timestamp(FIM), pd.Timestamp(SERIE_FIM))

fim_era5 = pd.Timestamp(era5.attrs["valid_time_stop"]) + pd.Timedelta(hours=23)
if pd.Timestamp(e5_fim) > fim_era5:
    print("ATENÇÃO: parte do período está no ERA5T (preliminar), sujeito a correções.")

e5 = era5[list(VARS_ERA5)].sel(
    time=slice(e5_inicio, e5_fim),
    latitude=slice(REGIAO["lat_n"], REGIAO["lat_s"]),            # norte → sul
    longitude=slice(lon_para_360(REGIAO["lon_o"]), lon_para_360(REGIAO["lon_l"])),
).rename(VARS_ERA5)

# A máscara terra-mar não muda no tempo: guardamos só o primeiro instante
lsm = e5["lsm"].isel(time=0).drop_vars("time")
e5 = e5.drop_vars("lsm")

checar_tamanho(e5, "ERA5 (recorte)")

Agora sim, carregamos os números na memória (`.load()`). Pode levar 1–3 minutos, porque cada instante lê um campo global da nuvem. Em seguida:

- convertemos a pressão de Pa para **hPa**;
- voltamos a longitude para −180..180, para plotar;
- salvamos o recorte em NetCDF, para não precisar baixar de novo.

In [ ]:
e5 = e5.load()
lsm = lsm.load()

e5["msl"] = e5["msl"] / 100.0
e5["msl"].attrs["units"] = "hPa"
e5["vel10"] = np.hypot(e5["u10"], e5["v10"])
e5["vel10"].attrs["units"] = "m s-1"

e5 = lon_para_180(e5)
lsm = lon_para_180(lsm)

e5.to_netcdf(PASTA_DADOS / "t1_era5_recorte.nc")
print(e5)

## 7. CMEMS: GLORYS12 (nível do mar) e WAVERYS (ondas)

Usamos `copernicusmarine.open_dataset`, que devolve um `xarray.Dataset` **preguiçoso** já recortado no servidor.

O CMEMS guarda cada conjunto em **duas organizações de chunks**:

| Serviço | Chunks | Bom para |
|---|---|---|
| `arco-geo-series` | pequenos no tempo, grandes no espaço | **mapas** (área grande, poucos instantes) |
| `arco-time-series` | grandes no tempo, pequenos no espaço | **séries temporais** (um ponto, muitos instantes) |

Para os mapas usamos `arco-geo-series`. Na seção 10 fazemos uma chamada separada com `arco-time-series` para a série no ponto.

**GLORYS12** (`cmems_mod_glo_phy_my_0.083deg_P1D-m`): reanálise oceânica global, 1/12° (\~8 km), 50 níveis, **médias diárias**. Variável: `zos` (altura da superfície do mar acima do geoide, m).

O aviso `WARNING ... depth dimension exceed the dataset coordinates` é esperado: pedimos 0–1 m e o primeiro nível do modelo está em 0,49 m.

In [ ]:
ID_GLORYS = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

glorys = copernicusmarine.open_dataset(
    dataset_id=ID_GLORYS,
    variables=["zos"],
    minimum_longitude=REGIAO["lon_o"], maximum_longitude=REGIAO["lon_l"],
    minimum_latitude=REGIAO["lat_s"], maximum_latitude=REGIAO["lat_n"],
    start_datetime=INICIO, end_datetime=FIM,
    service="arco-geo-series",                 # chunks otimizados para mapas
    coordinates_selection_method="outside",    # garante que a área pedida fique inteira
)
# "outside" pode trazer um dia a mais nas pontas: ficamos só com os dias da janela
glorys = glorys.sel(time=slice(INICIO[:10], FIM[:10]))
checar_tamanho(glorys, "GLORYS12 (recorte)")
glorys = glorys.load()
glorys.to_netcdf(PASTA_DADOS / "t1_glorys_recorte.nc")

for v in glorys.data_vars:
    print(f"{v:5s} [{glorys[v].attrs.get('units')}]  {glorys[v].attrs.get('standard_name')}")

**Anomalia do nível do mar.** O `zos` inclui a topografia dinâmica média (dezenas de cm, ligada às correntes de grande escala). Para destacar a resposta ao ciclone, subtraímos uma média de referência. Por padrão é a média do próprio período. Se `ZOS_REF_INICIO`/`ZOS_REF_FIM` estiverem preenchidos, usamos esse período mais longo (só `zos`, que é leve).

In [ ]:
if ZOS_REF_INICIO is None:
    zos_ref = glorys["zos"].mean("time")
    texto_ref = f"média de {INICIO[:10]} a {FIM[:10]}"
else:
    ref = copernicusmarine.open_dataset(
        dataset_id=ID_GLORYS, variables=["zos"],
        minimum_longitude=REGIAO["lon_o"], maximum_longitude=REGIAO["lon_l"],
        minimum_latitude=REGIAO["lat_s"], maximum_latitude=REGIAO["lat_n"],
        start_datetime=ZOS_REF_INICIO, end_datetime=ZOS_REF_FIM,
        service="arco-geo-series", coordinates_selection_method="outside",
    )
    checar_tamanho(ref, "GLORYS12 zos (referência)")
    zos_ref = ref["zos"].mean("time").load()
    texto_ref = f"média de {ZOS_REF_INICIO} a {ZOS_REF_FIM}"

glorys["zos_anom"] = glorys["zos"] - zos_ref
print("Referência da anomalia:", texto_ref)

**WAVERYS** (`cmems_mod_glo_wav_my_0.2deg_PT3H-i`): reanálise de ondas (modelo MFWAM), 0,2°, **a cada 3 h**. O ID foi confirmado com `copernicusmarine.describe(product_id="GLOBAL_MULTIYEAR_WAV_001_032")`. Variáveis:

| Nome | Significado | Unidade |
|---|---|---|
| `VHM0` | altura significativa (Hs) | m |
| `VTPK` | período de pico (Tp) | s |
| `VMDR` | direção média **de onde vêm** as ondas (convenção meteorológica/náutica, a partir do norte, sentido horário) | graus |
| `VHM0_WW` | Hs do **windsea** (ondas geradas pelo vento local) | m |
| `VHM0_SW1` | Hs do **swell** primário (ondas geradas longe) | m |
| `VMDR_SW1` | direção **de onde vem** o swell primário | graus |

In [ ]:
ID_WAVERYS = "cmems_mod_glo_wav_my_0.2deg_PT3H-i"

waverys = copernicusmarine.open_dataset(
    dataset_id=ID_WAVERYS,
    variables=["VHM0", "VTPK", "VMDR", "VHM0_WW", "VHM0_SW1"],
    minimum_longitude=REGIAO["lon_o"], maximum_longitude=REGIAO["lon_l"],
    minimum_latitude=REGIAO["lat_s"], maximum_latitude=REGIAO["lat_n"],
    start_datetime=INICIO, end_datetime=FIM,
    service="arco-geo-series",
    coordinates_selection_method="outside",
)
checar_tamanho(waverys, "WAVERYS (recorte)")
waverys = waverys.load()
waverys.to_netcdf(PASTA_DADOS / "t1_waverys_recorte.nc")

for v in waverys.data_vars:
    print(f"{v:9s} [{waverys[v].attrs.get('units')}]  {waverys[v].attrs.get('standard_name')}")

## 8. Situação sinótica (ERA5)

Sequência de mapas a cada `PASSO_MAPAS_H` horas com:

- **pressão ao nível do mar** em isolinhas a cada 4 hPa (os números são os valores em hPa);
- **vento a 10 m**: intensidade em cores e direção em vetores (reduzidos para \~20 por eixo).

Aqui **não** pintamos o continente, porque a gênese do ciclone pode ocorrer sobre a terra. A estrela vermelha marca o ponto da série temporal.

**O que observar:** onde e quando o centro de baixa pressão se forma, como se desloca, e quando o vento sobre a plataforma de Santos passa a soprar **de SW**, paralelo à costa.

In [ ]:
NCOL = 4   # >>> ALUNO: número de colunas da figura
nlin = int(np.ceil(len(DATAS_MAPAS) / NCOL))
niveis_vel = np.arange(0, 26, 2)
niveis_pnm = np.arange(940, 1048, 4)
n = passo_vetores(e5.sizes["longitude"])

fig, axs = plt.subplots(nlin, NCOL, figsize=(4.2 * NCOL, 3.6 * nlin),
                        subplot_kw=dict(projection=ccrs.PlateCarree()), squeeze=False)
for ax, t in zip(axs.flat, DATAS_MAPAS):
    c = e5.sel(time=t)
    mapa_base(ax, REGIAO, terra=False)
    cf = ax.contourf(c.longitude, c.latitude, c["vel10"], levels=niveis_vel,
                     cmap=cmocean.cm.speed, extend="max", transform=ccrs.PlateCarree())
    cs = ax.contour(c.longitude, c.latitude, c["msl"], levels=niveis_pnm,
                    colors="k", linewidths=0.7, transform=ccrs.PlateCarree())
    ax.clabel(cs, fmt="%d", fontsize=6)
    q = ax.quiver(c.longitude[::n], c.latitude[::n], c["u10"][::n, ::n], c["v10"][::n, ::n],
                  scale=250, width=0.0035, color="0.1", transform=ccrs.PlateCarree())
    ax.plot(PONTO["lon"], PONTO["lat"], marker="*", ms=11, color="red", mec="k",
            transform=ccrs.PlateCarree(), zorder=6)
    ax.set_title(f"PNM (hPa) e vento 10 m — {t:%Y-%m-%d %H} UTC", fontsize=8)
for ax in axs.flat[len(DATAS_MAPAS):]:
    ax.set_visible(False)

y = barra_cores(fig, cf, "Intensidade do vento a 10 m (m s$^{-1}$)")
ax.quiverkey(q, 0.85, y, 15, "15 m s$^{-1}$", labelpos="E", coordinates="figure")
rodape(fig, "Fonte: ERA5 (Hersbach et al., 2020), via ARCO-ERA5 (Google Cloud).")
salvar_figura(fig, f"t1_pnm_vento_{DATAS_MAPAS[0]:%Y-%m-%dT%H}_{DATAS_MAPAS[-1]:%Y-%m-%dT%H}.png")
plt.show()

## 9. Resposta oceânica

### 9.1 Nível do mar (GLORYS12, médias diárias)

Anomalia de `zos` (cores, paleta divergente centrada em zero). **O que observar:** sobre-elevação (vermelho) junto à costa quando o vento de SW empilha água na plataforma.

**Ressalvas importantes (leia antes de interpretar):**

- O GLORYS12 tem **médias diárias**. Os picos de maré meteorológica duram horas, então ficam **suavizados**. A grade de \~8 km também representa mal a plataforma rasa. **O `zos` serve para o padrão espacial e a ordem de grandeza, não para a altura do pico.** Para isso, compare com um marégrafo (ver "Para ir além").
- O GLORYS12 **não inclui maré** astronômica.

In [ ]:
datas_glorys = pd.DatetimeIndex(glorys.time.values)
NCOL = 4   # >>> ALUNO
nlin = int(np.ceil(len(datas_glorys) / NCOL))

lim = float(np.nanpercentile(np.abs(glorys["zos_anom"]), 99))
lim = np.ceil(lim * 20) / 20                     # arredonda para múltiplos de 5 cm
niveis_zos = np.linspace(-lim, lim, 21)

fig, axs = plt.subplots(nlin, NCOL, figsize=(4.2 * NCOL, 3.6 * nlin),
                        subplot_kw=dict(projection=ccrs.PlateCarree()), squeeze=False)
for ax, t in zip(axs.flat, datas_glorys):
    c = glorys.sel(time=t)
    mapa_base(ax, REGIAO, terra=True)
    cf = ax.contourf(c.longitude, c.latitude, c["zos_anom"], levels=niveis_zos,
                     cmap=cmocean.cm.balance, extend="both", transform=ccrs.PlateCarree())
    ax.plot(PONTO["lon"], PONTO["lat"], marker="*", ms=11, color="yellow", mec="k",
            transform=ccrs.PlateCarree(), zorder=6)
    ax.set_title(f"Anomalia de zos — {t:%Y-%m-%d} (média diária)", fontsize=8)
for ax in axs.flat[len(datas_glorys):]:
    ax.set_visible(False)

barra_cores(fig, cf, f"Anomalia de zos (m), referência: {texto_ref}")
rodape(fig, "Fonte: GLORYS12 (Lellouche et al., 2021), CMEMS GLOBAL_MULTIYEAR_PHY_001_030, via Copernicus Marine Toolbox (ARCO).")
salvar_figura(fig, f"t1_zos_{datas_glorys[0]:%Y-%m-%d}_{datas_glorys[-1]:%Y-%m-%d}.png")
plt.show()

### 9.2 Campo de ondas (WAVERYS)

Altura significativa **Hs** (cores) e **direção média** das ondas (setas).

**Convenção de direção.** O `VMDR` segue a convenção **meteorológica/náutica**: é a direção **de onde vêm** as ondas, em graus a partir do norte, no sentido horário (VMDR = 180° significa ondas vindas do sul). Mantemos essa convenção em todo o notebook. Para desenhar as setas no sentido em que as ondas se propagam, usamos a mesma fórmula que se usa para o vento em convenção meteorológica:

$$u = -\sin(\theta), \qquad v = -\cos(\theta)$$

O sinal negativo faz a seta apontar **para onde as ondas vão**. Exemplo: θ = 180° (de sul) dá u = 0, v = +1, uma seta para o norte. As setas têm comprimento fixo e mostram só a direção.

**O que observar:** a região de Hs máximo na pista de vento de SW atrás do ciclone, e a chegada de ondas de S/SW à costa de SP.

In [ ]:
datas_ondas = [t for t in DATAS_MAPAS if t in pd.DatetimeIndex(waverys.time.values)]
NCOL = 4   # >>> ALUNO
nlin = int(np.ceil(len(datas_ondas) / NCOL))
niveis_hs = np.arange(0, np.ceil(float(waverys["VHM0"].max())) + 0.5, 0.5)
n = passo_vetores(waverys.sizes["longitude"])

fig, axs = plt.subplots(nlin, NCOL, figsize=(4.2 * NCOL, 3.6 * nlin),
                        subplot_kw=dict(projection=ccrs.PlateCarree()), squeeze=False)
for ax, t in zip(axs.flat, datas_ondas):
    c = waverys.sel(time=t)
    teta = np.deg2rad(c["VMDR"])            # de onde vêm (convenção meteorológica)
    u_onda = -np.sin(teta)                  # seta apontando para onde as ondas vão
    v_onda = -np.cos(teta)
    mapa_base(ax, REGIAO, terra=True)
    cf = ax.contourf(c.longitude, c.latitude, c["VHM0"], levels=niveis_hs,
                     cmap=cmocean.cm.amp, transform=ccrs.PlateCarree())
    ax.quiver(c.longitude[::n], c.latitude[::n], u_onda[::n, ::n], v_onda[::n, ::n],
              scale=30, width=0.003, color="k", transform=ccrs.PlateCarree())
    ax.plot(PONTO["lon"], PONTO["lat"], marker="*", ms=11, color="cyan", mec="k",
            transform=ccrs.PlateCarree(), zorder=6)
    ax.set_title(f"Hs e direção média — {t:%Y-%m-%d %H} UTC", fontsize=8)
for ax in axs.flat[len(datas_ondas):]:
    ax.set_visible(False)

barra_cores(fig, cf, "Altura significativa Hs (m); setas: sentido de propagação (VMDR = de onde vêm)")
rodape(fig, "Fonte: WAVERYS (Law-Chune et al., 2021), CMEMS GLOBAL_MULTIYEAR_WAV_001_032, via Copernicus Marine Toolbox (ARCO).")
salvar_figura(fig, f"t1_hs_direcao_{datas_ondas[0]:%Y-%m-%dT%H}_{datas_ondas[-1]:%Y-%m-%dT%H}.png")
plt.show()

## 10. Série temporal no ponto

### 10.1 Dados no ponto

- **CMEMS:** nova chamada com `service="arco-time-series"` (chunks longos no tempo), numa caixa pequena em volta do ponto e no período `SERIE_INICIO`–`SERIE_FIM`. Perto da costa, a grade pode ter terra (NaN), então escolhemos o **ponto de oceano mais próximo**.
- **ERA5:** o armazenamento ARCO **não tem** uma versão organizada para séries temporais. Para poucas semanas de dados horários isso não é problema: já carregamos o recorte regional e extraímos o ponto dele. **Não peça anos de dados horários em um ponto**: cada hora lê um campo global, e a leitura fica muito lenta.

In [ ]:
CAIXA = 0.3   # meia-largura (graus) da caixa em volta do ponto

def serie_cmems(dataset_id, variaveis):
    """Abre uma série temporal do CMEMS numa caixa pequena em volta do PONTO."""
    return copernicusmarine.open_dataset(
        dataset_id=dataset_id, variables=variaveis,
        minimum_longitude=PONTO["lon"] - CAIXA, maximum_longitude=PONTO["lon"] + CAIXA,
        minimum_latitude=PONTO["lat"] - CAIXA, maximum_latitude=PONTO["lat"] + CAIXA,
        minimum_depth=0, maximum_depth=1,
        start_datetime=SERIE_INICIO, end_datetime=SERIE_FIM,
        service="arco-time-series",           # chunks otimizados para séries temporais
        coordinates_selection_method="outside",
    )

st_glorys = serie_cmems(ID_GLORYS, ["zos"]).load()
st_waverys = serie_cmems(ID_WAVERYS, ["VHM0", "VTPK", "VMDR", "VHM0_WW", "VHM0_SW1", "VMDR_SW1"]).load()

# Ponto de oceano mais próximo em cada grade
lon_g, lat_g = ponto_oceano_mais_proximo(st_glorys["zos"].isel(time=0), PONTO["lon"], PONTO["lat"])
lon_w, lat_w = ponto_oceano_mais_proximo(st_waverys["VHM0"].isel(time=0), PONTO["lon"], PONTO["lat"])
lon_e, lat_e = ponto_oceano_mais_proximo(lsm.where(lsm < 0.5), PONTO["lon"], PONTO["lat"])

pt_glorys = st_glorys.sel(longitude=lon_g, latitude=lat_g)
pt_waverys = st_waverys.sel(longitude=lon_w, latitude=lat_w)
pt_era5 = e5.sel(longitude=lon_e, latitude=lat_e).sel(time=slice(SERIE_INICIO, SERIE_FIM))

print(f"Ponto pedido:    {PONTO['lon']:.2f}, {PONTO['lat']:.2f}")
print(f"Ponto GLORYS12:  {lon_g:.2f}, {lat_g:.2f}")
print(f"Ponto WAVERYS:   {lon_w:.2f}, {lat_w:.2f}")
print(f"Ponto ERA5:      {lon_e:.2f}, {lat_e:.2f}")

### 10.2 Vento paralelo à costa

Com o ângulo da costa α (convenção da célula de parâmetros), a componente do vento paralela à costa é:

$$U_{\parallel} = u \sin\alpha + v \cos\alpha$$

U∥ > 0 indica vento soprando no sentido de α (de SW, em Santos), com transporte de Ekman **para a costa** no Hemisfério Sul.

In [ ]:
alfa = np.deg2rad(ANGULO_COSTA_GRAUS)
pt_era5["u_par"] = pt_era5["u10"] * np.sin(alfa) + pt_era5["v10"] * np.cos(alfa)

# Anomalia do zos no ponto em relação à média da série (o zos absoluto inclui a topografia média)
pt_glorys["zos_anom"] = pt_glorys["zos"] - pt_glorys["zos"].mean("time")

### 10.3 Figura

Painéis empilhados com eixo de tempo comum. A faixa cinza marca a janela do evento (`INICIO`–`FIM`).

- (a) PNM (ERA5, horária);
- (b) vento a 10 m em **vetores** (a seta aponta **para onde** o vento sopra; seta para cima = vento de sul), para ver a **inversão do vento** no ponto;
- (c) vento a 10 m: intensidade e componente paralela à costa;
- (d) anomalia de `zos` (GLORYS12, diária);
- (e) Hs e Tp (WAVERYS, 3-horário);
- (f) Hs do windsea vs. Hs do swell primário;
- (g) direção média das ondas (`VMDR`) e direção do swell primário (`VMDR_SW1`), ambas **de onde vêm** (convenção meteorológica).

**Perguntas para guiar a análise:** quando o vento inverte de NE para SW? O `zos` sobe quando U∥ fica positivo? Há defasagem? O máximo de Hs vem do windsea ou do swell? A direção das ondas gira junto com o vento?

In [ ]:
PASSO_VETORES_H = 6   # >>> ALUNO: intervalo entre os vetores de vento (horas)

fig, axs = plt.subplots(7, 1, figsize=(10, 16), sharex=True,
                        gridspec_kw={"height_ratios": [1, 0.6, 1, 1, 1, 1, 1]})

ax = axs[0]
ax.plot(pt_era5.time, pt_era5["msl"], color="k")
ax.set_ylabel("PNM (hPa)")
ax.set_title("(a) Pressão ao nível do mar — ERA5", loc="left", fontsize=9)

ax = axs[1]
pv = pt_era5.isel(time=slice(None, None, PASSO_VETORES_H))
t_num = mdates.date2num(pd.DatetimeIndex(pv.time.values))
# angles="uv": a direção da seta não depende da escala dos eixos
q = ax.quiver(t_num, np.zeros(len(t_num)), pv["u10"], pv["v10"], angles="uv",
              scale=120, width=0.002, color="k")
ax.quiverkey(q, 0.93, 1.08, 10, "10 m s$^{-1}$", labelpos="E", fontproperties={"size": 7})
ax.set_ylim(-1, 1)
ax.set_yticks([])
ax.set_title("(b) Vento a 10 m (vetores; seta aponta para onde o vento sopra) — ERA5", loc="left", fontsize=9)

ax = axs[2]
ax.plot(pt_era5.time, pt_era5["vel10"], color="k", label="intensidade")
ax.plot(pt_era5.time, pt_era5["u_par"], color="tab:blue",
        label=f"paralela à costa (α = {ANGULO_COSTA_GRAUS:.0f}°; > 0 = de SW)")
ax.axhline(0, color="0.5", lw=0.6)
ax.set_ylabel("Vento 10 m (m s$^{-1}$)")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("(c) Intensidade e componente paralela à costa — ERA5", loc="left", fontsize=9)

ax = axs[3]
# a média diária representa o dia inteiro: plotamos no meio do dia (12 UTC)
ax.plot(pt_glorys.time + pd.Timedelta(hours=12), pt_glorys["zos_anom"], "o-", color="tab:red", ms=3)
ax.axhline(0, color="0.5", lw=0.6)
ax.set_ylabel("Anomalia de zos (m)")
ax.set_title("(d) Nível do mar: anomalia de zos — GLORYS12 (média diária)", loc="left", fontsize=9)

ax = axs[4]
ax.plot(pt_waverys.time, pt_waverys["VHM0"], color="k", label="Hs")
ax.set_ylabel("Hs (m)")
ax2 = ax.twinx()
ax2.plot(pt_waverys.time, pt_waverys["VTPK"], color="tab:green", lw=0.9, label="Tp")
ax2.set_ylabel("Tp (s)", color="tab:green")
ax.set_title("(e) Altura significativa e período de pico — WAVERYS", loc="left", fontsize=9)

ax = axs[5]
ax.plot(pt_waverys.time, pt_waverys["VHM0_WW"], color="tab:orange", label="windsea")
ax.plot(pt_waverys.time, pt_waverys["VHM0_SW1"], color="tab:blue", label="swell primário")
ax.set_ylabel("Hs (m)")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("(f) Windsea vs. swell — WAVERYS", loc="left", fontsize=9)

ax = axs[6]
ax.plot(pt_waverys.time, pt_waverys["VMDR"], "o", color="k", ms=3, label="direção média (VMDR)")
ax.plot(pt_waverys.time, pt_waverys["VMDR_SW1"], "^", color="tab:blue", ms=3,
        label="direção do swell primário (VMDR_SW1)")
ax.set_ylim(0, 360)
ax.set_yticks([0, 90, 180, 270, 360])
ax.set_yticklabels(["N", "L", "S", "O", "N"])
ax.set_ylabel("De onde vêm")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("(g) Direção das ondas (convenção meteorológica: de onde vêm) — WAVERYS", loc="left", fontsize=9)

for ax in axs:
    ax.axvspan(pd.Timestamp(INICIO), pd.Timestamp(FIM), color="0.9", zorder=0)
    ax.grid(lw=0.3)
axs[-1].set_xlabel("Tempo (UTC)")
axs[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axs[-1].xaxis.get_major_locator()))
fig.suptitle(f"{PONTO['nome']} ({PONTO['lon']:.2f}°, {PONTO['lat']:.2f}°)", y=0.995)
fig.tight_layout()
rodape(fig, "Fontes: ERA5 (Hersbach et al., 2020); GLORYS12 (Lellouche et al., 2021); WAVERYS (Law-Chune et al., 2021).")
salvar_figura(fig, f"t1_serie_ponto_{SERIE_INICIO[:10]}_{SERIE_FIM[:10]}.png")
plt.show()

## 11. Para ir além

Sugestões para enriquecer o seu estudo de caso:

- **Marégrafo:** compare o `zos` com o nível do mar observado na região de Santos. Fontes possíveis: **UHSLC/GLOSS** (https://uhslc.soest.hawaii.edu) e o **BNDO** da Marinha do Brasil. Remova a maré astronômica (por exemplo, com análise harmônica ou um filtro passa-baixa de \~40 h) para obter a maré meteorológica observada, com resolução horária.
- **Boia de ondas:** compare Hs, Tp e direção com o **PNBoia** (Marinha do Brasil, boia de Santos).
- **Altimetria por satélite:** produtos L3/L4 de nível do mar do CMEMS (`SEALEVEL_GLO_PHY_L4_MY_008_047`).
- **Vento por satélite:** escaterômetros (ASCAT) para validar o vento do ERA5 sobre o mar.
- **Outro modelo:** compare com um modelo regional de maré meteorológica, ou com a reanálise de ondas ERA5 (que também traz Hs e direção).
- **Trajetória do ciclone:** localize o mínimo de PNM em cada instante e trace o caminho do centro.
- **Pista e tempo de atuação do vento:** estime a pista (*fetch*) e a duração do vento de SW e compare com a altura de onda esperada.

## Opcional: copiar dados e figuras para o Google Drive

A pasta `/content` some ao fim da sessão do Colab. Para guardar os resultados, mude `COPIAR_PARA_DRIVE` para `True` e rode a célula. O Colab vai pedir autorização para acessar o seu Drive.

In [ ]:
COPIAR_PARA_DRIVE = False   # >>> ALUNO: True para copiar
PASTA_NO_DRIVE = "ACA0430_trabalho1"   # >>> ALUNO: nome da pasta dentro de "Meu Drive"

if EM_COLAB and COPIAR_PARA_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = Path("/content/drive/MyDrive") / PASTA_NO_DRIVE
    shutil.copytree(PASTA_DADOS, destino / "dados", dirs_exist_ok=True)
    shutil.copytree(PASTA_FIGURAS, destino / "figuras", dirs_exist_ok=True)
    print("Copiado para", destino)

## 12. Referências

**Dados e infraestrutura**
- Hersbach, H., et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, 146, 1999–2049. https://doi.org/10.1002/qj.3803
- Lellouche, J.-M., et al. (2021). The Copernicus Global 1/12° Oceanic and Sea Ice GLORYS12 Reanalysis. *Frontiers in Earth Science*, 9, 698876. https://doi.org/10.3389/feart.2021.698876
- Law-Chune, S., Aouf, L., Dalphinet, A., Levier, B., Drillet, Y., & Drevillon, M. (2021). WAVERYS: a CMEMS global wave reanalysis during the altimetry period. *Ocean Dynamics*, 71, 357–378. https://doi.org/10.1007/s10236-020-01433-w
- Abernathey, R. P., et al. (2021). Cloud-native repositories for big scientific data. *Computing in Science & Engineering*, 23(2), 26–35. https://doi.org/10.1109/MCSE.2021.3059437
- Documentação técnica (não revisada por pares, citar como fonte dos dados): repositório ARCO-ERA5 (https://github.com/google-research/arco-era5); Manual do Usuário CMEMS-GLO-PUM-001-030; página do produto GLOBAL_MULTIYEAR_WAV_001_032 (DOI 10.48670/moi-00022); documentação do Copernicus Marine Toolbox (https://toolbox-docs.marine.copernicus.eu).

**Tema**
- Campos, R. M., Camargo, R., & Harari, J. (2010). Caracterização de eventos extremos do nível do mar em Santos e sua correspondência com as reanálises do modelo do NCEP no sudoeste do Atlântico Sul. *Revista Brasileira de Meteorologia*, 25(2), 175–184. https://doi.org/10.1590/S0102-77862010000200003
- Gramcianinov, C. B., Campos, R. M., Guedes Soares, C., & de Camargo, R. (2020). Extreme waves generated by cyclonic winds in the western portion of the South Atlantic Ocean. *Ocean Engineering*, 213, 107745. https://doi.org/10.1016/j.oceaneng.2020.107745